# 버그 찾기 1 (쉬움): MM은 정말 실패했나

위치 힌트: **MM 셀**(4번째 코드 셀) 하나에 있다. 결론 중 틀린 문장은 여러 개일 수 있다.

데이터: cities를 흉내 낸 **합성** 활성값이다. 도시 400개 × (참 문장 1, 거짓 문장 1), d=64. 진짜 Llama 활성값이 아니다.

**하는 법**
1. 위에서부터 끝까지 실행하고, 맨 아래 **결론**을 읽는다.
2. 코드나 설명 중 "말과 실제 동작이 다른 곳"을 찾아 **문장으로** 적는다(맨 아래 `내 답` 셀).
3. 고치면 숫자가 어떻게 바뀔지 **한 줄 예측**을 먼저 적는다.
4. 고쳐서 다시 돌린다. 문법이 막히면 고칠 내용을 문장으로 쓰고 `???`로 남겨 두면 Claude가 채운다.

이 노트북은 오류 없이 끝까지 돈다. 정답은 Claude에게 따로 있다.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

d = np.load("acts/p1.npz")
X, y, group = d["X"], d["y"], d["group"]
print(X.shape, "참 비율", y.mean(), "도시 수", len(np.unique(group)))

In [ ]:
def group_split(groups, fractions=(0.6, 0.2, 0.2), seed=0):
    """같은 group의 행이 서로 다른 split에 들어가지 않게 나눈다 (probe_lab.group_split과 같다)."""
    unique = np.unique(groups)
    rng = np.random.default_rng(seed)
    rng.shuffle(unique)
    a = int(round(fractions[0] * len(unique)))
    b = int(round(fractions[1] * len(unique)))
    parts = [unique[:a], unique[a : a + b], unique[a + b :]]
    return [np.flatnonzero(np.isin(groups, p)) for p in parts]


tr, va, te = group_split(group)
print(len(tr), len(va), len(te))

## MM probe

01 노트북과 같은 정의다. 방향은 θ = μ₁ − μ₀, 결정 경계는 두 class 평균의 **중점**을 지난다. 점수가 0보다 크면 참으로 판정한다.

In [ ]:
mu1 = X[tr][y[tr] == 1].mean(0)
mu0 = X[tr][y[tr] == 0].mean(0)
theta = mu1 - mu0

score_mm = X[te] @ theta  # θ 방향 투영 점수
acc_mm = np.mean((score_mm > 0) == y[te])

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=2000).fit(X[tr], y[tr])
acc_lr = lr.score(X[te], y[te])

In [ ]:
print(f"MM test 정확도: {acc_mm:.2f}")
print(f"LR test 정확도: {acc_lr:.2f}")

## 결론

1. MM의 test 정확도는 0.50이다. 동전 던지기 수준이다.
2. 따라서 평균 차 방향(θ)에는 참/거짓 정보가 거의 없다.
3. LR은 0.93이다. 정보는 공분산을 보정한 방향에만 있다.
4. 이 데이터에서는 MM probe를 버리고 LR을 쓴다.

## 내 답

- 말과 실제 동작이 다른 곳:
- 그래서 틀린 결론 문장(번호):
- 고치면 어떻게 바뀔지 예측:
- 고친 뒤 실제 결과: